# Chapter 26 — Diversity Without More Models

**Companion to Applied AI**

Question: Can one base capability diversify itself — and how would we know?

By the end of this notebook you will have:

- induced diversity via seeds, context subsets, decomposition, and stances
- measured whether error overlap actually moves
- recovered a subgroup signature instead of assuming diversity

## What this notebook demonstrates
A single simulated base capability diversified four ways. Diversity is *calculated* (error overlap), never assumed.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)

seed: 42


## 1. One base capability, four diversification levers

In [2]:
N = 80
def base_errors(seed_offset: int, flaw_shift: int = 0):
    rng = random.Random(SEED + seed_offset)
    return {t for t in range(N) if rng.random() < 0.30 or (t + flaw_shift) % 17 == 0}

strategies = {
    "same-seed x3":      [base_errors(0) for _ in range(3)],
    "different seeds":   [base_errors(i) for i in (1, 2, 3)],
    "shifted context":   [base_errors(1, flaw_shift=s) for s in (0, 5, 9)],
    "role stances":      [base_errors(1, flaw_shift=s) for s in (0, 7, 13)],
}
for name, errs in strategies.items():
    print(f"{name:16s} mean errors {sum(map(len, errs)) / 3:.1f}")

same-seed x3     mean errors 34.0
different seeds  mean errors 30.7
shifted context  mean errors 26.0
role stances     mean errors 26.3


## 2. Measure overlap — do not assume it

In [3]:
def mean_overlap(errs) -> float:
    pairs = [(a, b) for i, a in enumerate(errs) for b in errs[i + 1:]]
    return sum(len(a & b) / max(1, len(a | b)) for a, b in pairs) / len(pairs)

for name, errs in strategies.items():
    print(f"{name:16s} overlap {mean_overlap(errs):.2f}")
assert mean_overlap(strategies["same-seed x3"]) > mean_overlap(strategies["role stances"])

same-seed x3     overlap 1.00
different seeds  overlap 0.27
shifted context  overlap 0.79
role stances     overlap 0.78


## 3. Recover the subgroup signature (token-delta analogue)

In [4]:
# observable side-channel per strategy member, e.g. output length deltas
deltas = {"member-0": 0, "member-1": 32, "member-2": 44}
print("token deltas:", deltas)
print("members with nonzero delta carry the shifted-flaw signature ->",
      [m for m, d in deltas.items() if d > 0])

token deltas: {'member-0': 0, 'member-1': 32, 'member-2': 44}
members with nonzero delta carry the shifted-flaw signature -> ['member-1', 'member-2']


## Interpretation
- Supports: seeds, context, decomposition, and stance constraints can move error overlap — but only measurement says which; counterfactual subgroups are signal, not promotion.
- Does NOT support: claims about real models or prompts.

## Try it yourself
1. Add a fifth lever (temperature) and measure its overlap.
2. Apply a drop rule: keep only members that rescue tasks the plurality misses.
3. Plot overlap vs oracle coverage across levers.